In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from collections import Counter
import math

In [3]:
df = pd.read_parquet("20240610.parquet")

print(df.shape)
df.head()

(731512, 37)


,DST_IP,DST_IP_SUBNET,DST_IP_VERSION,DST_ASN,DST_COUNTRY,DST_PORT,PROTOCOL,TIME_FIRST,TIME_LAST,DURATION,...,QUIC_TLS_EXT_TYPE,QUIC_PACKETS,PPI,PPI_LEN,PPI_DURATION,PPI_ROUNDTRIPS,PHIST_SRC_SIZES,PHIST_DST_SIZES,PHIST_SRC_IPT,PHIST_DST_IPT
0,7a756769d0,ae12b508c7,4,15169,US,443,17,2024-06-09 23:00:00+02:00,2024-06-09 23:00:10.219832+02:00,10.219832,...,"[0, 43, 17513, 10, 65037, 57, 16, 45, 13, 27, ...","[129, 130, 130, 130, 130, 130, 133, 128, 128, ...","[[0, 3, 3, 0, 0, 0, 9, 0, 2, 0, 15, 0, 0, 0, 1...",30,0.091,3,"[0, 19, 45, 4, 12, 6, 4, 7]","[0, 52, 1, 1, 20, 2, 7, 34]","[51, 22, 13, 1, 2, 2, 4, 1]","[77, 13, 13, 3, 3, 2, 4, 1]"
1,75d7b68d14,ae12b508c7,4,15169,US,443,17,2024-06-09 23:00:00+02:00,2024-06-09 23:00:47.877138+02:00,47.877138,...,"[27, 17513, 57, 43, 65037, 0, 13, 10, 51, 16, ...","[129, 130, 130, 133, 128, 128, 128, 133, 132, ...","[[0, 3, 3, 9, 0, 0, 0, 8, 0, 0, 0, 1, 4, 21, 0...",30,1.196,8,"[0, 4, 19, 2, 3, 0, 6, 8]","[0, 30, 3, 7, 1, 1, 1, 1]","[18, 11, 3, 2, 2, 0, 2, 3]","[20, 7, 8, 2, 1, 0, 2, 3]"
2,f2ac2651bf,ae12b508c7,4,15169,US,443,17,2024-06-09 23:00:00+02:00,2024-06-09 23:00:02.924524+02:00,2.924524,...,"[16, 10, 43, 13, 45, 27, 51, 65037, 17513, 42,...","[129, 130, 133, 128, 128, 128, 133, 132, 128, ...","[[0, 0, 15, 0, 0, 11, 8, 1, 0, 0, 0, 1, 0, 0, ...",30,0.107,6,"[0, 5, 4, 3, 0, 2, 3, 27]","[0, 23, 0, 2, 5, 1, 1, 1]","[33, 0, 5, 3, 1, 0, 0, 1]","[20, 5, 3, 2, 1, 0, 0, 1]"
3,c7a666cd46,ae12b508c7,4,15169,US,443,17,2024-06-09 23:00:00+02:00,2024-06-09 23:00:00.253800+02:00,0.253800,...,"[13, 0, 27, 57, 17513, 45, 10, 16, 51, 43, 650...","[129, 130, 133, 128, 128, 128, 133, 132, 128, ...","[[0, 1, 14, 0, 0, 11, 5, 1, 0, 3, 0, 0, 1, 0, ...",25,0.254,5,"[0, 2, 3, 2, 0, 1, 1, 3]","[0, 5, 0, 2, 1, 0, 2, 3]","[8, 1, 1, 0, 1, 0, 0, 0]","[10, 0, 1, 0, 1, 0, 0, 0]"
4,199026ef4f,04c9bd8530,6,15169,BE,443,17,2024-06-09 23:00:00+02:00,2024-06-09 23:00:00.078434+02:00,0.078434,...,"[0, 23, 65281, 10, 16, 5, 34, 51, 42, 43, 13, ...","[131, 130, 130, 130, 133, 128, 128, 132, 128, ...","[[0, 1, 0, 0, 15, 0, 0, 11, 14, 0, 3, 12, 0, 0...",19,0.079,4,"[0, 2, 2, 1, 0, 0, 1, 3]","[0, 3, 1, 2, 0, 0, 2, 2]","[5, 3, 0, 0, 0, 0, 0, 0]","[7, 2, 0, 0, 0, 0, 0, 0]"


In [4]:
ppi_features = df[
    [
        "PPI",
        "PPI_LEN",
        "PPI_DURATION",
        "PPI_ROUNDTRIPS"
    ]
].copy()

ppi_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 731512 entries, 0 to 731511
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PPI             731512 non-null  object 
 1   PPI_LEN         731512 non-null  int64  
 2   PPI_DURATION    731512 non-null  float64
 3   PPI_ROUNDTRIPS  731512 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 22.3+ MB


In [5]:
# Calculate the average inter-packet time for each flow
ppi_features["mean_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[0])
)

In [6]:
# Calculate the variability of inter-packet times
ppi_features["std_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[0])
)

In [7]:
# Calculate the average packet size
ppi_features["mean_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[2])
)

In [8]:
# Calculate packet size variability
ppi_features["std_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[2])
)

In [9]:
def direction_change_ratio(directions):

    # If there is only one packet, no direction change is possible
    if len(directions) < 2:
        return 0

    changes = 0

    # Compare every packet direction with the next one
    for i in range(len(directions) - 1):

        if directions[i] != directions[i + 1]:
            changes += 1

    # Return the proportion of direction changes
    return changes / (len(directions) - 1)

In [10]:
ppi_features["direction_change_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: direction_change_ratio(x[1])
    )
)

In [11]:
def forward_packet_ratio(directions):

    if len(directions) == 0:
        return 0

    forward_packets = np.sum(directions == 1)

    return forward_packets / len(directions)

In [12]:
ppi_features["forward_packet_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: forward_packet_ratio(x[1])
    )
)

In [13]:
ppi_features["log_ppi_duration"] = np.log1p(ppi_features["PPI_DURATION"])
ppi_features["log_mean_ipt"] = np.log1p(ppi_features["mean_ipt"])
ppi_features["log_std_ipt"] = np.log1p(ppi_features["std_ipt"])

In [14]:
ppi_final = ppi_features[
    [
        # Original Features
        "PPI_LEN",
        "PPI_ROUNDTRIPS",

        # Log-transformed Features
        "log_ppi_duration",
        "log_mean_ipt",
        "log_std_ipt",

        # Engineered Features
        "mean_packet_size",
        "std_packet_size",
        "direction_change_ratio",
        "forward_packet_ratio"
    ]
].copy()

print("Final PPI Feature Bank Shape:", ppi_final.shape)

ppi_final.head()

Final PPI Feature Bank Shape: (731512, 9)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio
0,30,3,0.087095,1.394593,1.844657,305.066667,448.216609,0.206897,0.533333
1,30,8,0.786638,3.710315,5.200603,285.400000,426.120609,0.517241,0.500000
2,30,6,0.101654,1.518784,2.285496,551.833333,553.884530,0.379310,0.533333
3,25,5,0.226338,2.412336,3.506581,437.080000,515.110700,0.416667,0.480000
4,19,4,0.076035,1.640528,1.912475,485.894737,551.395635,0.444444,0.473684


In [15]:
cid_features = df[
    [
        "DST_ASN",
        "QUIC_OCCID",
        "QUIC_OSCID",
        "QUIC_SCID",
        "QUIC_RETRY_SCID",
        "QUIC_SNI"
    ]
].copy()

cid_features.head()

,DST_ASN,QUIC_OCCID,QUIC_OSCID,QUIC_SCID,QUIC_RETRY_SCID,QUIC_SNI
0,15169,,861cf811876afa2d,e61cf811876afa2d,,www.google.com
1,15169,,a0ece0eff7678405,e0ece0eff7678405,,pagead2.googlesyndication.com
2,15169,,56beb0d3a7bae298,f6beb0d3a7bae298,,play.googleapis.com
3,15169,,b53c742b89dd1093,f53c742b89dd1093,,play-fe.googleapis.com
4,15169,9cf640,2538a79124393ec34e4eb9332ab0,e538a79124393ec3,,accounts.google.com


In [16]:
cid_features["occid_length"] = cid_features["QUIC_OCCID"].str.len()

cid_features["oscid_length"] = cid_features["QUIC_OSCID"].str.len()

cid_features["scid_length"] = cid_features["QUIC_SCID"].str.len()

cid_features["retry_length"] = cid_features["QUIC_RETRY_SCID"].str.len()

cid_features["sni_length"] = cid_features["QUIC_SNI"].str.len()

cid_features["sni_labels"] = (
    cid_features["QUIC_SNI"]
    .str.split(".")
    .str.len()
)

In [17]:
cid_features["retry_present"] = (cid_features["retry_length"] > 0).astype(int)

In [18]:
def shannon_entropy(text):

    if len(text) == 0:
        return 0.0

    counts = Counter(text)

    entropy = 0.0

    length = len(text)

    for count in counts.values():

        p = count / length

        entropy -= p * math.log2(p)

    return entropy

In [19]:
def normalized_entropy(text):

    if len(text) == 0:
        return 0.0

    H = shannon_entropy(text)

    Hmax = math.log2(min(len(text), 16))

    return H / Hmax if Hmax > 0 else 0.0

In [20]:
cid_features["occid_entropy"] = (cid_features["QUIC_OCCID"].apply(normalized_entropy))

cid_features["oscid_entropy"] = (cid_features["QUIC_OSCID"].apply(normalized_entropy))

cid_features["scid_entropy"] = (cid_features["QUIC_SCID"].apply(normalized_entropy))

In [21]:
cid_features[
    [
        "occid_entropy",
        "oscid_entropy",
        "scid_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
occid_entropy,731512.0,0.244100,0.399967,0.000000,0.00000,0.000000,0.742098,1.0
oscid_entropy,731512.0,0.816347,0.059746,0.480949,0.78125,0.812500,0.863205,1.0
scid_entropy,731512.0,0.771972,0.169349,0.000000,0.75766,0.800705,0.843750,1.0


In [22]:
MIN_FLOWS = max(50, int(0.001 * len(cid_features)))

In [23]:
asn_groups = cid_features.groupby("DST_ASN")

In [24]:
asn_profile = asn_groups.agg({

    "scid_length": ["count", "median"],

    "scid_entropy": "median",

    "oscid_length": "median",

    "oscid_entropy": "median",

    "retry_present": "mean",

    "sni_length": "median",

    "sni_labels": "median"

})

In [25]:
asn_profile.columns = [

    "flows",
    "median_scid_length",
    "median_scid_entropy",
    "median_oscid_length",
    "median_oscid_entropy",
    "retry_rate",
    "median_sni_length",
    "median_sni_labels",

]

In [26]:
asn_profile = asn_profile[asn_profile["flows"] >= MIN_FLOWS]

In [27]:
asn_profile.head()

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
6185,988,40.0,0.925925,16.0,0.800705,0.000000,17.0,3.0
8075,9792,28.0,0.884787,16.0,0.800705,0.000511,21.0,3.0
13335,15288,40.0,0.904613,16.0,0.820160,0.000000,15.0,3.0
15169,562622,16.0,0.800705,16.0,0.814141,0.000000,20.0,3.0
16509,3844,40.0,0.919392,16.0,0.825566,0.000000,18.0,3.0


In [28]:
cid_features = cid_features.merge(
    asn_profile,
    on="DST_ASN",
    how="left"
)

In [29]:
cid_features["known_asn"] = (
    cid_features["flows"].notna().astype(int)
)

In [30]:
global_scid_length = cid_features["scid_length"].median()

global_scid_entropy = cid_features["scid_entropy"].median()

global_oscid_length = cid_features["oscid_length"].median()

global_oscid_entropy = cid_features["oscid_entropy"].median()

global_retry_rate = cid_features["retry_present"].mean()

global_sni_length = cid_features["sni_length"].median()

global_sni_labels = cid_features["sni_labels"].median()

In [31]:
fill_values = {
    "median_scid_length": global_scid_length,
    "median_scid_entropy": global_scid_entropy,
    "median_oscid_length": global_oscid_length,
    "median_oscid_entropy": global_oscid_entropy,
    "retry_rate": global_retry_rate,
    "median_sni_length": global_sni_length,
    "median_sni_labels": global_sni_labels
}

cid_features = cid_features.fillna(fill_values)

In [32]:
asn_profile.describe().T

,count,mean,std,min,25%,50%,75%,max
flows,12.0,60792.750000,159498.870895,988.000000,3014.750000,5767.0000,24030.000000,562622.000000
median_scid_length,12.0,28.500000,11.571910,16.000000,16.000000,31.0000,40.000000,40.000000
median_scid_entropy,12.0,0.863173,0.064274,0.769455,0.800705,0.8947,0.920768,0.926550
median_oscid_length,12.0,16.000000,0.000000,16.000000,16.000000,16.0000,16.000000,16.000000
median_oscid_entropy,12.0,0.814019,0.008848,0.800705,0.812500,0.8125,0.815646,0.831955
retry_rate,12.0,0.000044,0.000147,0.000000,0.000000,0.0000,0.000000,0.000511
median_sni_length,12.0,20.333333,5.959459,13.000000,16.500000,19.0000,22.000000,33.000000
median_sni_labels,12.0,3.083333,0.288675,3.000000,3.000000,3.0000,3.000000,4.000000


In [33]:
asn_profile.sort_values("flows", ascending=False).head(20)

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
15169,562622,16.0,0.800705,16.0,0.814141,0.000000,20.0,3.0
32934,68441,16.0,0.769455,16.0,0.812500,0.000000,18.0,3.0
396982,50256,16.0,0.800705,16.0,0.812500,0.000020,29.0,3.0
13335,15288,40.0,0.904613,16.0,0.820160,0.000000,15.0,3.0
8075,9792,28.0,0.884787,16.0,0.800705,0.000511,21.0,3.0
54113,7421,34.0,0.911428,16.0,0.812500,0.000000,20.0,4.0
20940,4113,16.0,0.788910,16.0,0.812500,0.000000,25.0,3.0
16509,3844,40.0,0.919392,16.0,0.825566,0.000000,18.0,3.0
36183,3361,40.0,0.924895,16.0,0.812500,0.000000,15.0,3.0


In [34]:
cid_features["scid_length_deviation"] = (
    cid_features["scid_length"] -
    cid_features["median_scid_length"]
).abs()

cid_features["oscid_length_deviation"] = (
    cid_features["oscid_length"] -
    cid_features["median_oscid_length"]
).abs()

In [35]:
cid_features["sni_length_deviation"] = (
    cid_features["sni_length"] -
    cid_features["median_sni_length"]
).abs()

In [36]:
cid_features["scid_entropy_deviation"] = (
    cid_features["scid_entropy"] -
    cid_features["median_scid_entropy"]
).abs()

cid_features["oscid_entropy_deviation"] = (
    cid_features["oscid_entropy"] -
    cid_features["median_oscid_entropy"]
).abs()

In [37]:
cid_final = cid_features[
    [
        # Raw CID Features
        "occid_length",
        "oscid_length",
        "scid_length",
        "retry_present",
        "sni_length",
        "sni_labels",

        # Statistical Features
        "oscid_entropy",
        "scid_entropy",

        # Context Features
        "scid_length_deviation",
        "oscid_length_deviation",
        "scid_entropy_deviation",
        "oscid_entropy_deviation",
        "sni_length_deviation",
        "known_asn"
    ]
].copy()


print(cid_final.shape)

(731512, 14)


In [38]:
cid_final.head()

,occid_length,oscid_length,scid_length,retry_present,sni_length,sni_labels,oscid_entropy,scid_entropy,scid_length_deviation,oscid_length_deviation,scid_entropy_deviation,oscid_entropy_deviation,sni_length_deviation,known_asn
0,0,16,16,0,14,3,0.757660,0.800705,0.0,0.0,0.000000,0.056481,6.0,1
1,0,16,16,0,29,3,0.788910,0.738205,0.0,0.0,0.062500,0.025231,9.0,1
2,0,16,16,0,19,3,0.863205,0.863205,0.0,0.0,0.062500,0.049064,1.0,1
3,0,16,16,0,22,3,0.875000,0.906250,0.0,0.0,0.105545,0.060859,2.0,1
4,6,28,16,0,19,3,0.857827,0.812500,0.0,12.0,0.011795,0.043686,1.0,1


In [39]:
hist_features = df[
    [
        "PHIST_SRC_SIZES",
        "PHIST_DST_SIZES",
        "PHIST_SRC_IPT",
        "PHIST_DST_IPT"
    ]
].copy()

In [40]:
def histogram_entropy(hist):

    # Convert to NumPy array
    hist = np.array(hist, dtype=float)

    # Total observations
    total = hist.sum()

    # Handle empty histograms
    if total == 0:
        return 0

    # Convert counts to probabilities
    probabilities = hist / total

    # Calculate Shannon entropy
    entropy = 0

    for p in probabilities:
        if p > 0:
            entropy -= p * math.log2(p)

    # Normalize entropy
    max_entropy = math.log2(len(hist))

    return entropy / max_entropy

In [41]:
# Source packet size histogram entropy
hist_features["src_size_entropy"] = (
    hist_features["PHIST_SRC_SIZES"]
    .apply(histogram_entropy)
)

# Destination packet size histogram entropy
hist_features["dst_size_entropy"] = (
    hist_features["PHIST_DST_SIZES"]
    .apply(histogram_entropy)
)

# Source IPT histogram entropy
hist_features["src_ipt_entropy"] = (
    hist_features["PHIST_SRC_IPT"]
    .apply(histogram_entropy)
)

# Destination IPT histogram entropy
hist_features["dst_ipt_entropy"] = (
    hist_features["PHIST_DST_IPT"]
    .apply(histogram_entropy)
)

In [42]:
hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
src_size_entropy,731512.0,0.566865,0.182985,0.0,0.506809,0.607689,0.688749,0.917811
dst_size_entropy,731512.0,0.578053,0.205242,0.0,0.516265,0.639432,0.709362,0.935785
src_ipt_entropy,731512.0,0.350120,0.196283,0.0,0.226092,0.333333,0.486383,0.983355
dst_ipt_entropy,731512.0,0.349089,0.216313,0.0,0.197224,0.343941,0.506349,1.000000


In [43]:
hist_final = hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].copy()

print(hist_final.shape)

hist_final.head()

(731512, 4)


,src_size_entropy,dst_size_entropy,src_ipt_entropy,dst_ipt_entropy
0,0.749714,0.644860,0.641144,0.566845
1,0.726207,0.519711,0.740142,0.732438
2,0.611467,0.493065,0.391472,0.574968
3,0.819716,0.711313,0.425871,0.272230
4,0.732387,0.748813,0.318145,0.254735


In [44]:
flow_df = df.copy()

In [45]:
flow_df["duration_safe"] = flow_df["DURATION"].clip(lower=1e-6)

In [46]:
flow_df["total_bytes"] = (
    flow_df["BYTES"] +
    flow_df["BYTES_REV"]
)

In [47]:
flow_df["total_packets"] = (
    flow_df["PACKETS"] +
    flow_df["PACKETS_REV"]
)

In [48]:
flow_df["byte_rate"] = (
    flow_df["total_bytes"] /
    flow_df["duration_safe"]
)

In [49]:
flow_df["packet_rate"] = (
    flow_df["total_packets"] /
    flow_df["duration_safe"]
)

In [50]:
flow_df["avg_packet_size"] = (
    flow_df["total_bytes"] /
    flow_df["total_packets"].clip(lower=1)
)

In [51]:
flow_df["byte_ratio"] = (
    flow_df["BYTES"] /
    flow_df["BYTES_REV"].clip(lower=1)
)

In [52]:
flow_df["packet_ratio"] = (
    flow_df["PACKETS"] /
    flow_df["PACKETS_REV"].clip(lower=1)
)

In [53]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
]

In [54]:
flow_features.replace([np.inf, -np.inf], np.nan, inplace=True)

flow_features.isnull().sum()

DURATION           0
FLOW_END_REASON    0
total_bytes        0
total_packets      0
byte_rate          0
packet_rate        0
avg_packet_size    0
byte_ratio         0
packet_ratio       0
dtype: int64

In [55]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
].copy()

In [56]:
flow_features["log_duration"] = np.log1p(flow_features["DURATION"])

flow_features["log_total_bytes"] = np.log1p(flow_features["total_bytes"])

flow_features["log_total_packets"] = np.log1p(flow_features["total_packets"])

flow_features["log_byte_rate"] = np.log1p(flow_features["byte_rate"])

flow_features["log_packet_rate"] = np.log1p(flow_features["packet_rate"])

flow_features["log_byte_ratio"] = np.log1p(flow_features["byte_ratio"])

flow_features["log_packet_ratio"] = np.log1p(flow_features["packet_ratio"])

In [57]:
log_columns = [

"log_duration",

"FLOW_END_REASON",

"log_total_bytes",

"log_total_packets",

"log_byte_rate",

"log_packet_rate",

"avg_packet_size",

"log_byte_ratio",

"log_packet_ratio"

]

flow_features[log_columns].corr()

,log_duration,FLOW_END_REASON,log_total_bytes,log_total_packets,log_byte_rate,log_packet_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
log_duration,1.000000,0.028680,0.530018,0.596895,-0.784975,-0.812380,0.022361,-0.049223,-0.185323
FLOW_END_REASON,0.028680,1.000000,0.001747,0.010223,-0.022433,-0.017403,-0.022754,0.004504,0.012233
log_total_bytes,0.530018,0.001747,1.000000,0.964597,-0.085183,-0.184748,0.458516,-0.276291,-0.411220
log_total_packets,0.596895,0.010223,0.964597,1.000000,-0.206884,-0.264749,0.225881,-0.236376,-0.324619
log_byte_rate,-0.784975,-0.022433,-0.085183,-0.206884,1.000000,0.980317,0.373306,0.006184,-0.060387
log_packet_rate,-0.812380,-0.017403,-0.184748,-0.264749,0.980317,1.000000,0.209246,0.048244,0.018879
avg_packet_size,0.022361,-0.022754,0.458516,0.225881,0.373306,0.209246,1.000000,-0.186364,-0.466119
log_byte_ratio,-0.049223,0.004504,-0.276291,-0.236376,0.006184,0.048244,-0.186364,1.000000,0.604878
log_packet_ratio,-0.185323,0.012233,-0.411220,-0.324619,-0.060387,0.018879,-0.466119,0.604878,1.000000


In [58]:
selected_columns = [

    "log_duration",
    "FLOW_END_REASON",
    "log_total_bytes",
    "log_byte_rate",
    "avg_packet_size",
    "log_byte_ratio",
    "log_packet_ratio"

]

flow_features_selected = flow_features[selected_columns].copy()

flow_features_selected.head()

,log_duration,FLOW_END_REASON,log_total_bytes,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
0,2.417683,1,11.259220,8.935008,362.574766,0.307033,0.603802
1,3.889310,1,9.977621,6.111156,250.441860,1.381015,0.670158
2,1.367245,1,10.682927,9.609840,566.285714,2.195140,0.847298
3,0.226179,1,9.361171,10.732316,465.080000,0.590181,0.653926
4,0.075510,1,9.224736,11.770143,533.894737,0.742657,0.641854


In [59]:
quic_features = df[
    [
        "QUIC_VERSION",
        "QUIC_CLIENT_VERSION",
        "QUIC_TOKEN_LENGTH",
        "QUIC_ZERO_RTT",
        "QUIC_MULTIPLEXED"
    ]
].copy()

quic_features.head()

,QUIC_VERSION,QUIC_CLIENT_VERSION,QUIC_TOKEN_LENGTH,QUIC_ZERO_RTT,QUIC_MULTIPLEXED
0,1,1,70,5,0
1,1,1,71,2,0
2,1,1,70,1,0
3,1,1,71,1,0
4,1,1,70,4,0


In [60]:
quic_features["token_present"] = (
    quic_features["QUIC_TOKEN_LENGTH"] > 0
).astype(int)

In [61]:
quic_features["log_token_length"] = np.log1p(
    quic_features["QUIC_TOKEN_LENGTH"]
)

In [62]:
quic_features["zero_rtt_present"] = (
    quic_features["QUIC_ZERO_RTT"] > 0
).astype(int)

In [63]:
quic_features["multiplexed"] = (
    quic_features["QUIC_MULTIPLEXED"] > 0
).astype(int)

In [64]:
version_map = {
    version: idx
    for idx, version in enumerate(
        sorted(quic_features["QUIC_VERSION"].unique())
    )
}

quic_features["quic_version_id"] = (
    quic_features["QUIC_VERSION"].map(version_map)
)

In [65]:
version_map

{np.int64(0): 0,
 np.int64(1): 1,
 np.int64(3467641594): 2,
 np.int64(4207849474): 3,
 np.int64(4207849486): 4,
 np.int64(4207849491): 5,
 np.int64(4278190109): 6}

In [66]:
quic_features["log_zero_rtt"] = np.log1p(
    quic_features["QUIC_ZERO_RTT"]
)

In [67]:
quic_features["log_multiplexed"] = np.log1p(
    quic_features["QUIC_MULTIPLEXED"]
)

In [68]:
quic_selected = quic_features[
    [
        "quic_version_id",
        "log_token_length",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
]

quic_selected.corr()

,quic_version_id,log_token_length,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
quic_version_id,1.000000,0.121553,-0.069284,-0.081155,-0.044836,-0.047093
log_token_length,0.121553,1.000000,0.649336,0.730052,-0.010449,-0.013238
log_zero_rtt,-0.069284,0.649336,1.000000,0.888818,-0.028025,-0.050342
zero_rtt_present,-0.081155,0.730052,0.888818,1.000000,-0.062244,-0.076218
log_multiplexed,-0.044836,-0.010449,-0.028025,-0.062244,1.000000,0.951102
multiplexed,-0.047093,-0.013238,-0.050342,-0.076218,0.951102,1.000000


In [69]:
selected_quic = quic_features[
    [
        "quic_version_id",
        "QUIC_TOKEN_LENGTH",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
].copy()

selected_quic.head()

,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,1,70,1.791759,1,0.0,0
1,1,71,1.098612,1,0.0,0
2,1,70,0.693147,1,0.0,0
3,1,71,0.693147,1,0.0,0
4,1,70,1.609438,1,0.0,0


In [70]:
with_CID_df = pd.concat([ppi_final,cid_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(with_CID_df.shape)

with_CID_df.to_parquet(
    "with_CID.parquet",
    index=False
)

(731512, 40)


In [71]:
check_df = pd.read_parquet("with_CID.parquet")

print(check_df.shape)
check_df.head()

(731512, 40)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,occid_length,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,30,3,0.087095,1.394593,1.844657,305.066667,448.216609,0.206897,0.533333,0,...,8.935008,362.574766,0.307033,0.603802,1,70,1.791759,1,0.0,0
1,30,8,0.786638,3.710315,5.200603,285.400000,426.120609,0.517241,0.500000,0,...,6.111156,250.441860,1.381015,0.670158,1,71,1.098612,1,0.0,0
2,30,6,0.101654,1.518784,2.285496,551.833333,553.884530,0.379310,0.533333,0,...,9.609840,566.285714,2.195140,0.847298,1,70,0.693147,1,0.0,0
3,25,5,0.226338,2.412336,3.506581,437.080000,515.110700,0.416667,0.480000,0,...,10.732316,465.080000,0.590181,0.653926,1,71,0.693147,1,0.0,0
4,19,4,0.076035,1.640528,1.912475,485.894737,551.395635,0.444444,0.473684,6,...,11.770143,533.894737,0.742657,0.641854,1,70,1.609438,1,0.0,0


In [72]:
without_CID_df = pd.concat([ppi_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(without_CID_df.shape)

without_CID_df.to_parquet(
    "without_CID.parquet",
    index=False
)

(731512, 26)


In [73]:
check_df2 = pd.read_parquet("without_CID.parquet")

print(check_df2.shape)
check_df2.head()

(731512, 26)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,src_size_entropy,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,30,3,0.087095,1.394593,1.844657,305.066667,448.216609,0.206897,0.533333,0.749714,...,8.935008,362.574766,0.307033,0.603802,1,70,1.791759,1,0.0,0
1,30,8,0.786638,3.710315,5.200603,285.400000,426.120609,0.517241,0.500000,0.726207,...,6.111156,250.441860,1.381015,0.670158,1,71,1.098612,1,0.0,0
2,30,6,0.101654,1.518784,2.285496,551.833333,553.884530,0.379310,0.533333,0.611467,...,9.609840,566.285714,2.195140,0.847298,1,70,0.693147,1,0.0,0
3,25,5,0.226338,2.412336,3.506581,437.080000,515.110700,0.416667,0.480000,0.819716,...,10.732316,465.080000,0.590181,0.653926,1,71,0.693147,1,0.0,0
4,19,4,0.076035,1.640528,1.912475,485.894737,551.395635,0.444444,0.473684,0.732387,...,11.770143,533.894737,0.742657,0.641854,1,70,1.609438,1,0.0,0


In [74]:
print(with_CID_df.columns.duplicated().sum())
print(without_CID_df.columns.duplicated().sum())

0
0
